## HOUSE PRICES

# Data Loading

In [1]:
import pandas as pd
test = pd.read_csv(r"test.csv")
test.head()

FileNotFoundError: [Errno 2] No such file or directory: 'test.csv'

In [ ]:
train = pd.read_csv("train.csv")
train.head()

train.csv` dibaca menggunakan library Pandas dan disimpan ke dalam variabel train

Fungsi `head()` digunakan untuk menampilkan 5 baris pertama dari dataset

# Data Cleaning 


In [ ]:
train.info()

In [ ]:
train.describe(include="all")

Fungsi `info()` digunakan untuk melihat jumlah data, nama kolom, tipe data, dan jumlah data yang tidak kosong pada setiap kolom.

Fungsi `describe(include="all")` digunakan untuk melihat statistik deskriptif dari seluruh kolom, baik kolom numerik maupun kategorikal.

In [ ]:
missing_values = train.isnull().sum()
missing_values[missing_values > 0]

kolom missing value buat mencari tahu kolom mana yang datanya kosong dan berapa jumlahnya.

In [ ]:
less = missing_values[missing_values < 1000].index
over = missing_values[missing_values >= 1000].index

In [ ]:
numeric_features = train[less].select_dtypes(include=['number']).columns
train[numeric_features] = train[numeric_features].fillna(train[numeric_features].median())

In [ ]:
kategorical_features = train[less].select_dtypes(include=['object']).columns

for column in kategorical_features:
    train[column] = train[column].fillna(train[column].mode()[0])

In [ ]:
df = train.drop(columns=over)

In [ ]:
missing_values = df.isnull().sum()    
missing_values[missing_values > 0]

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

for features in numeric_features:
    plt.figure(figsize=(10,6))
    sns.boxplot(x=df[features])
    plt.title(f'Box Plot of {features}')
    plt.show()

In [ ]:
Q1 = df[numeric_features].quantile(0.25)
Q3 = df[numeric_features].quantile(0.75)
IQR = Q3 - Q1

In [ ]:
condition = ~((df[numeric_features] < (Q1 - 1.5 * IQR)) | 
              (df[numeric_features] > (Q3 + 1.5 * IQR))).any(axis=1)

df_filtered_numeric = df.loc[condition, numeric_features]

categorical_numeric = df.select_dtypes(include=['object']).columns

df = pd.concat([df_filtered_numeric, df.loc[condition, categorical_numeric]], axis=1)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df[numeric_features] = scaler.fit_transform(df[numeric_features])


In [ ]:
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.histplot(train[numeric_features[3]], kde=True)
plt.title("Histogram Sebelum Standarisasi")

In [ ]:
plt.subplot(1, 2, 2)
sns.histplot(df[numeric_features[3]], kde=True)
plt.title("Histogram Setelah Standardisasi")

In [ ]:
duplicates = df.duplicated()

print("Baris duplikat:")
print(df[duplicates])

In [ ]:
 df = df.drop_duplicates()

print("DataFrame setelah menghapus duplikat:")
print(df)

In [ ]:
from sklearn.preprocessing import LabelEncoder


label_encoder = LabelEncoder()
df_lencoder = pd.DataFrame(df)

In [ ]:
category_features = df.select_dtypes(include=['object']).columns
df[category_features]

In [ ]:
df_one_hot = pd.get_dummies(df, columns=category_features)
df_one_hot

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Inisialisasi LabelEncoder
label_encoder = LabelEncoder()
df_lencoder = pd.DataFrame(df)

for col in category_features:
    df_lencoder[col] = label_encoder.fit_transform(df[col])

# Menampilkan hasil
df_lencoder


# EDA EXDA

In [ ]:
df_lencoder.head()

In [ ]:
# Menghitung jumlah dan persentase missing values di setiap kolom
missing_values = df_lencoder.isnull().sum()
missing_percentage = (missing_values / len(df_lencoder)) * 100

missing_data = pd.DataFrame({
    'Missing Values': missing_values,
    'Percentage': missing_percentage
}).sort_values(by='Missing Values', ascending=False)

# Menampilkan kolom yang memiliki missing values saja
missing_data[missing_data['Missing Values'] > 0]


In [ ]:
import matplotlib.pyplot as plt

# Menghitung jumlah variabel/kolom
num_vars = df_lencoder.shape[1]

# Menentukan jumlah baris dan kolom untuk grid subplot
n_cols = 4  # Jumlah kolom yang diinginkan
n_rows = -(-num_vars // n_cols)  # Ceiling division untuk menentukan jumlah baris

# Membuat subplot
fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, n_rows * 4))

# Flatten axes array untuk memudahkan iterasi
axes = axes.flatten()

# Plot setiap variabel menggunakan perulangan
for i, column in enumerate(df_lencoder.columns):
    df_lencoder[column].hist(ax=axes[i], bins=20, edgecolor='black')
    axes[i].set_title(column)
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Frequency')

# Menghapus jendela subplot sisa yang tidak terpakai
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

# Menyesuaikan layout agar tidak tumpang tindih
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualisasi distribusi data untuk beberapa kolom
columns_to_plot = ['OverallQual', 'YearBuilt', 'LotArea', 'SaleType', 'SaleCondition']

plt.figure(figsize=(15, 10))
for i, column in enumerate(columns_to_plot, 1):
    plt.subplot(2, 3, i)
    sns.histplot(df_lencoder[column], kde=True, bins=30)
    plt.title(f'Distribution of {column}')

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualisasi korelasi antar variabel numerik
plt.figure(figsize=(12, 10))
correlation_matrix = df_lencoder.corr()

# Membuat heatmap korelasi
sns.heatmap(correlation_matrix, annot=False, cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Correlation Matrix')
plt.show()


In [ ]:
# Menghitung korelasi antara variabel target dan semua variabel lainnya
target_corr = df_lencoder.corr()['SalePrice']
# (Opsional) Mengurutkan hasil korelasi berdasarkan korelasi
target_corr_sorted = target_corr.abs().sort_values(ascending=False)
plt.figure(figsize=(10, 6))
target_corr_sorted.plot(kind='bar')
plt.title(f'Correlation with SalePrice')
plt.xlabel('Variables')
plt.ylabel('Correlation Coefficient')
plt.show()

## DATA SPLITTING

In [ ]:
import sklearn

In [ ]:
# Misahin fitur x dan target y
X = df_lencoder.drop(columns=['SalePrice'])
y = df_lencoder['SalePrice']

- **X:** kumpulan atribut atau fitur dari dataset, yaitu semua kolom kecuali kolom target (pada kasus ini SalePrice).
- **y:** target atau label yang akan kita prediksi merupakan kolom yang ingin kita pisahkan (pada kasus ini SalePrice).

In [ ]:
from sklearn.model_selection import train_test_split
# membagi dataset menjadi training dan testing 
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)


1. **X_train:** fitur untuk train set.
2. **X_test:** fitur untuk test set.
3. **y_train:** target untuk train set.
4. **y_test:** target untuk test set.

In [ ]:
# menghitung panjang/jumlah data 
print("Jumlah data: ",len(X))
# menghitung panjang/jumlah data pada x_train
print("Jumlah data latih: ",len(x_train))
# menghitung panjang/jumlah data pada x_test
print("Jumlah data test: ",len(x_test))

## Modelling

In [ ]:
# Melatih model 1 dengan algoritma Least Angle Regression
from sklearn import linear_model

lars = linear_model.Lars(n_nonzero_coefs=1).fit(x_train, y_train)

# Melatih model 2 dengan algoritma Linear Regression
from sklearn.linear_model import LinearRegression

LR = LinearRegression().fit(x_train, y_train)

# Melatih model 3 dengan algoritma Gradient Boosting Regressor
from sklearn.ensemble import GradientBoostingRegressor

GBR = GradientBoostingRegressor(random_state=184)
GBR.fit(x_train, y_train)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Evaluasi pada model LARS
pred_lars = lars.predict(x_test)

mae_lars = mean_absolute_error(y_test, pred_lars)
mse_lars = mean_squared_error(y_test, pred_lars)
r2_lars = r2_score(y_test, pred_lars)

# Membuat dictionary untuk menyimpan hasil evaluasi
data = {
    'MAE': [mae_lars],
    'MSE': [mse_lars],
    'R2': [r2_lars]
}

# Konversi dictionary menjadi DataFrame
df_results = pd.DataFrame(data, index=['Lars'])

df_results

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Evaluasi pada model LARS
pred_lars = lars.predict(x_test)

mae_lars = mean_absolute_error(y_test, pred_lars)
mse_lars = mean_squared_error(y_test, pred_lars)
r2_lars = r2_score(y_test, pred_lars)

# Membuat dictionary untuk menyimpan hasil evaluasi
data = {
    'MAE': [mae_lars],
    'MSE': [mse_lars],
    'R2': [r2_lars]
}

# Konversi dictionary menjadi DataFrame
df_results = pd.DataFrame(data, index=['Lars'])

df_results

In [ ]:
# Evaluasi pada model Linear Regression
pred_LR = LR.predict(x_test)
mae_LR = mean_absolute_error(y_test, pred_LR)
mse_LR = mean_squared_error(y_test, pred_LR)
r2_LR = r2_score(y_test, pred_LR)
# Menambahkan hasil evaluasi LR ke DataFrame
df_results.loc['Linear Regression'] = [mae_LR, mse_LR, r2_LR]
df_results


In [ ]:
# Evaluasi pada model Gradient Boosting Regressor
pred_GBR = GBR.predict(x_test)
mae_GBR = mean_absolute_error(y_test, pred_GBR)
mse_GBR = mean_squared_error(y_test, pred_GBR)
r2_GBR = r2_score(y_test, pred_GBR)
# Menambahkan hasil evaluasi GBR ke DataFrame
df_results.loc['GradientBoostingRegressor'] = [mae_GBR, mse_GBR, r2_GBR]
df_results

Berdasarkan hasil evaluasi yang diperoleh, model Gradient Boosting Regressor memiliki hasil paling baik dibandingkan model LARS dan Linear Regression. Model ini memperoleh nilai MAE sebesar 0,217360 dan MSE sebesar 0,089055, yang merupakan nilai kesalahan paling rendah di antara ketiga model. Selain itu, nilai R² yang diperoleh sebesar 0,889065 menunjukkan bahwa model mampu menjelaskan sekitar 88,91% variasi pada data target.

Sementara itu, model Linear Regression juga menunjukkan hasil yang cukup baik dengan nilai R² sebesar 0,880690. Berbeda dengan kedua model tersebut, model LARS memiliki nilai R² sebesar -0,018517, yang menunjukkan bahwa model ini belum mampu memberikan prediksi yang baik pada data pengujian.

Dari hasil tersebut, dapat disimpulkan bahwa Gradient Boosting Regressor merupakan model terbaik dari ketiga model yang diuji, karena memiliki nilai kesalahan paling rendah dan nilai R² paling tinggi. Meskipun demikian, hasil evaluasi ini tetap berdasarkan data pengujian yang digunakan, sehingga model masih dapat dikembangkan agar menghasilkan prediksi yang lebih akurat.

In [ ]:
import joblib
# Menyimpan model ke dalam file
joblib.dump(GBR, 'gbr_model.joblib')

In [ ]:
import pickle

# Menyimpan model ke dalam file
with open('gbr_model.pkl', 'wb') as file:
    pickle.dump(GBR, file)

## DEPLOYMENT

In [8]:
!pip install joblib
!pip install pickle


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Could not find a version that satisfies the requirement pickle (from versions: none)

[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for pickle


In [9]:
python.exe -m pip install --upgrade pip

SyntaxError: invalid syntax (842801469.py, line 1)

In [5]:
import joblib
import pickle

# Memuat model dari file joblib
joblib_model = joblib.load('gbr_model.joblib')

# Memuat model dari file pickle
with open('gbr_model.pkl', 'rb') as file:
    pickle_model = pickle.load(file)

ModuleNotFoundError: No module named 'joblib'